# Your First AI Employee: Building a Personalized Social Media Agent
Run the cells from top to bottom. Each section writes one module of the agent into the `agent/` folder, then runs it.
You need an OpenAI API key. In Colab, open the key icon in the left sidebar, add a secret named `OPENAI_API_KEY`, and switch on notebook access.

## 0. Setup

In [ ]:
!pip -q install openai openai-agents feedparser requests numpy pandas streamlit
%load_ext autoreload
%autoreload 2

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
os.makedirs("agent", exist_ok=True); os.makedirs("sources", exist_ok=True)

Smoke test: one API call. If this prints a reply, your key works.

In [ ]:
from openai import OpenAI
r = OpenAI().chat.completions.create(model="gpt-5.4-mini", messages=[{"role": "user", "content": "Say hello in five words."}])
print(r.choices[0].message.content)

## 1. Settings, model wrappers and memory
Three small modules. `config.py` holds the model name and platform rules. `llm.py` wraps the API. `memory.py` stores everything in one JSON file.

In [ ]:
%%writefile agent/__init__.py
"""A personal AI social media manager."""

In [ ]:
%%writefile agent/config.py
"""Central settings. Change the model here or with the SMM_MODEL environment variable."""
import os

MODEL = os.getenv("SMM_MODEL", "gpt-5.4-mini")
EMBED_MODEL = "text-embedding-3-small"
DATA_DIR = os.getenv("SMM_DATA_DIR", "data")
SOURCES_DIR = os.getenv("SMM_SOURCES_DIR", "sources")

# One entry per platform. Add "threads" or "bluesky" here to support more.
PLATFORMS = {
    "linkedin": {
        "max_chars": 3000,
        "comment_max_chars": 600,
        "rules": "Open with a hook in the first two lines. Use short paragraphs. "
                 "Use at most three hashtags, at the end.",
    },
    "x": {
        "max_chars": 280,
        "comment_max_chars": 280,
        "rules": "One idea per post. Hashtags only if they fit naturally.",
    },
}

# Edit this dictionary to follow the sources you care about.
FEEDS = {
    "techcrunch_ai": "https://techcrunch.com/category/artificial-intelligence/feed/",
    "arxiv_cs_ai": "http://export.arxiv.org/rss/cs.AI",
    "mit_tech_review": "https://www.technologyreview.com/feed/",
}

In [ ]:
%%writefile agent/llm.py
"""Thin wrappers around the OpenAI API. Every other module calls these."""
import json
import re

from openai import OpenAI

from .config import EMBED_MODEL, MODEL

_client = None


def client():
    """Create the client on first use, so the API key can be set after import."""
    global _client
    if _client is None:
        _client = OpenAI()
    return _client


def chat(messages, **kwargs):
    response = client().chat.completions.create(model=MODEL, messages=messages, **kwargs)
    return response.choices[0].message.content.strip()


def extract_json(text):
    """Parse the first JSON object or array found in a model reply."""
    match = re.search(r"[\[{].*[\]}]", text, re.S)
    if not match:
        raise ValueError("No JSON found in reply: " + text[:200])
    return json.loads(match.group(0))


def chat_json(messages):
    """Ask for a JSON object. The word JSON must appear in the messages."""
    text = chat(messages, response_format={"type": "json_object"})
    return extract_json(text)


def embed(texts, batch=100):
    vectors = []
    for i in range(0, len(texts), batch):
        response = client().embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch])
        vectors.extend(d.embedding for d in response.data)
    return vectors

In [ ]:
%%writefile agent/memory.py
"""Persistent memory: one JSON file you can open and read."""
import copy
import json
import os
import time
import uuid

from .config import DATA_DIR

PATH = os.path.join(DATA_DIR, "memory.json")
EMB_PATH = os.path.join(DATA_DIR, "embeddings.json")

EMPTY = {
    "field": "",          # what you write about
    "profile": "",        # headline, summary, positions
    "style": {},          # style profile learned from your writing
    "favorite_words": [],
    "banned_words": [],
    "corpus": [],         # your past posts and comments
    "drafts": [],         # everything the agent wrote
    "feedback": [],       # your edits and rejections
    "metrics": [],        # impressions, reactions, comments
    "lessons": [],        # rules the agent learned from feedback and metrics
}


def _read(path, default):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return default


def load():
    mem = _read(PATH, {})
    for key, value in EMPTY.items():
        mem.setdefault(key, copy.deepcopy(value))
    return mem


def save(mem):
    os.makedirs(DATA_DIR, exist_ok=True)
    with open(PATH, "w", encoding="utf-8") as f:
        json.dump(mem, f, indent=2, ensure_ascii=False)


def add(mem, key, item):
    item.setdefault("id", uuid.uuid4().hex[:8])
    item.setdefault("ts", time.strftime("%Y-%m-%d %H:%M"))
    mem[key].append(item)
    save(mem)
    return item


def find(mem, key, item_id):
    return next((x for x in mem[key] if x["id"] == item_id), None)


def load_embeddings():
    return _read(EMB_PATH, {})


def save_embeddings(embeddings):
    os.makedirs(DATA_DIR, exist_ok=True)
    with open(EMB_PATH, "w", encoding="utf-8") as f:
        json.dump(embeddings, f)

## 2. Style learning
First give the agent your writing. Choose one:
- **LinkedIn export:** request your data archive (Settings, Data privacy, Get a copy of your data), then upload `Shares.csv`, `Comments.csv`, `Profile.csv` and `Positions.csv`.
- **Your own text:** upload `.txt` files with posts separated by a line containing only `---`.
- **Neither yet:** run the sample cell to use the example author.

In [ ]:
from google.colab import files
uploaded = files.upload()   # skip this cell if you want the sample data
for name, data in uploaded.items():
    open(f"sources/{name}", "wb").write(data)

In [ ]:
%%writefile sources/sample_posts.txt
Shipping a model is the easy part. Keeping it correct for six months is the job.

We pushed a ranking model last spring and the offline metrics looked great. Three weeks later, click-through had dropped 4 percent and nobody could say why.

The cause was a feature that quietly changed units upstream. We now alert on feature distributions before we alert on accuracy.

Monitor inputs first. Outputs only tell you after the damage is done.
---
I interviewed 30 ML engineers this year. The strongest candidates all did one thing: they asked what the model does when it is wrong.

Most people prepare to explain architectures. Few prepare to explain failure modes.

If you want to stand out, bring one failure from your own work and walk through how you found it, what it cost, and what you changed.
---
Small teams do not need a bigger model. They need a smaller loop.

Our best improvement this quarter came from cutting evaluation time from two days to twenty minutes. We ran four times more experiments and found the bug that three bigger models had been hiding.

Speed of feedback beats size of model. Fix the loop first.
---
Hot take from our last planning meeting: write the eval before you write the prompt.

If you cannot say how you will tell a good output from a bad one, you are not ready to build. You are ready to experiment, which is fine, but call it that.

Ten test cases on a spreadsheet beat zero test cases and a clever prompt.
---
A founder asked me how to hire the first ML engineer. My answer was short.

Hire someone who has shipped something ugly to real users and fixed it at 2 a.m. Skip the person with the perfect research record and no production scars.

You can teach papers. It is much harder to teach judgment.
---
We removed 40 percent of our prompt last week and quality went up.

Every sentence we had added over six months was a patch for a bug we no longer had. Prompts grow like old codebases. Nobody deletes anything.

Schedule a prompt cleanup the way you schedule a refactor.
---
People ask whether AI will replace engineers. I see a different pattern on my team.

The engineers who write clear specs get the most out of coding assistants. The ones who struggle to describe the problem struggle with the tools too.

The skill that matters more every month is explaining what you want, precisely.
---
Three questions I ask before adding an agent to any workflow:
1. Can a simple script do this?
2. What happens when the agent is wrong?
3. Who reviews the output, and how fast?

If I cannot answer all three, I do not build it yet.

In [ ]:
%%writefile sources/sample_comments.txt
Agree on monitoring inputs first. We learned that the hard way with a unit change upstream.
---
Good point on feedback loops. What did you cut first to get evaluation down to minutes?
---
This matches what I see with small teams. Fast iteration beats a bigger model almost every time.
---
Curious how you handle the cases where the reviewer disagrees with the metric. That is where it gets messy for us.
---
Clear and practical. The part about writing the eval before the prompt is worth printing out.
---
I would add one thing: log the failures you fix, not only the ones you find. The list teaches you more than the dashboard.

In [ ]:
%%writefile agent/style.py
"""Learn the author's voice from past writing, and retrieve similar examples later."""
import csv
import hashlib
import os
import re
import uuid

import numpy as np

from . import memory
from .config import SOURCES_DIR
from .llm import chat_json, embed

TEXT_COLUMNS = ["sharecommentary", "commentary", "message", "text", "post"]
MIN_CHARS = {"post": 40, "comment": 20}


def _find_file(folder, name):
    """Case-insensitive file lookup, because LinkedIn exports use names like Profile.csv."""
    for f in os.listdir(folder):
        if f.lower() == name:
            return os.path.join(folder, f)
    return None


def _read_csv(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            yield {(k or "").strip().lower().replace(" ", ""): (v or "").strip() for k, v in row.items()}


def load_texts(folder=SOURCES_DIR):
    """Return [{'text', 'kind', 'source'}] for every post or comment found in the folder."""
    items = []
    if not os.path.isdir(folder):
        return items
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        low = name.lower()
        kind = "comment" if "comment" in low else "post"
        if low.endswith((".txt", ".md")):
            with open(path, encoding="utf-8") as f:
                chunks = f.read().split("\n---\n")
        elif low.endswith(".csv") and low not in ("profile.csv", "positions.csv"):
            chunks = [next((row[c] for c in TEXT_COLUMNS if row.get(c)), "") for row in _read_csv(path)]
        else:
            continue
        for chunk in chunks:
            text = chunk.strip()
            if len(text) >= MIN_CHARS[kind]:
                items.append({"text": text, "kind": kind, "source": low})
    return items


def load_profile(folder=SOURCES_DIR):
    """Build a short background text from Profile.csv and Positions.csv, if present."""
    parts = []
    if not os.path.isdir(folder):
        return ""
    path = _find_file(folder, "profile.csv")
    if path:
        for row in _read_csv(path):
            parts += [row.get("headline", ""), row.get("summary", "")]
    path = _find_file(folder, "positions.csv")
    if path:
        for row in list(_read_csv(path))[:6]:
            parts.append(f"{row.get('title', '')} at {row.get('companyname', '')}. {row.get('description', '')}")
    return "\n".join(p for p in parts if p.strip())


def ingest(mem, folder=SOURCES_DIR):
    """Add new texts from the folder to memory. Safe to run again after adding files."""
    known = {c["hash"] for c in mem["corpus"]}
    new = []
    for item in load_texts(folder):
        item["hash"] = hashlib.sha1(item["text"].encode()).hexdigest()[:12]
        if item["hash"] in known:
            continue
        known.add(item["hash"])
        item["id"] = uuid.uuid4().hex[:8]
        new.append(item)
    if new:
        vectors = embed([n["text"] for n in new])
        embeddings = memory.load_embeddings()
        embeddings.update({n["id"]: v for n, v in zip(new, vectors)})
        memory.save_embeddings(embeddings)
        mem["corpus"].extend(new)
    profile = load_profile(folder)
    if profile:
        mem["profile"] = profile
    memory.save(mem)
    return len(new)


def build_profile(mem, max_posts=40, max_comments=15):
    """Ask the model to describe how the author writes. The result is stored as JSON."""
    posts = [c["text"] for c in mem["corpus"] if c["kind"] == "post"][-max_posts:]
    comments = [c["text"] for c in mem["corpus"] if c["kind"] == "comment"][-max_comments:]
    prompt = (
        "Analyze the writing samples below and describe how this author writes.\n"
        "Return JSON with these keys: voice (two sentences), sentence_length (short, medium or long, "
        "with a typical word count), openers (three typical ways posts begin), structure (how a typical "
        "post is organized), signature_phrases (list), vocabulary_level, emoji_use, formatting "
        "(line breaks, lists, hashtags), avoid (things the author never does).\n"
        "Describe only patterns that appear in the samples.\n\n"
        "POSTS:\n" + "\n\n---\n\n".join(posts) + "\n\nCOMMENTS:\n" + "\n\n---\n\n".join(comments)
    )
    mem["style"] = chat_json([{"role": "user", "content": prompt}])
    memory.save(mem)
    return mem["style"]


def retrieve_examples(mem, query, k=4, kind="post"):
    """Return the k past texts most similar to the query, using cosine similarity."""
    embeddings = memory.load_embeddings()
    items = [c for c in mem["corpus"] if c["kind"] == kind and c["id"] in embeddings]
    if not items:
        return []
    q = np.array(embed([query])[0])
    matrix = np.array([embeddings[c["id"]] for c in items])
    sims = matrix @ q / (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q))
    return [items[i]["text"] for i in np.argsort(-sims)[:k]]


def set_words(mem, favorite=None, banned=None):
    if favorite is not None:
        mem["favorite_words"] = [w.strip() for w in favorite if w.strip()]
    if banned is not None:
        mem["banned_words"] = [w.strip() for w in banned if w.strip()]
    memory.save(mem)


def banned_hits(text, banned):
    """Return the banned words or symbols that appear in the text."""
    return [w for w in banned if re.search(r"(?<!\w)" + re.escape(w) + r"(?!\w)", text, re.I)]

In [ ]:
from agent import memory, style
mem = memory.load()
mem["field"] = "applied machine learning and AI startups"   # change this to your field
print("new items:", style.ingest(mem))
print(style.build_profile(mem))

In [ ]:
style.set_words(mem, favorite=["practical", "ship"], banned=["synergy", "game-changer", "\u2014"])
mem["banned_words"]

## 3. Topic discovery with tool calling
The model decides which tools to call. Your code runs them and sends the results back.

In [ ]:
%%writefile agent/tools.py
"""Tools the research agent can call. Each is a plain Python function plus a JSON spec."""
import json

import feedparser
import requests

from . import memory
from .config import FEEDS


def get_hacker_news(limit=15):
    """Current Hacker News front page stories."""
    r = requests.get(
        "https://hn.algolia.com/api/v1/search",
        params={"tags": "front_page", "hitsPerPage": limit},
        timeout=15,
    )
    r.raise_for_status()
    return [
        {
            "title": h["title"],
            "url": h.get("url") or f"https://news.ycombinator.com/item?id={h['objectID']}",
            "points": h.get("points"),
            "comments": h.get("num_comments"),
        }
        for h in r.json()["hits"]
    ]


def get_rss(feed, limit=10):
    """Latest entries from one of the feeds configured in config.FEEDS."""
    if feed not in FEEDS:
        return {"error": f"Unknown feed. Choose one of {list(FEEDS)}"}
    parsed = feedparser.parse(FEEDS[feed])
    return [
        {"title": e.get("title"), "url": e.get("link"), "summary": (e.get("summary") or "")[:300]}
        for e in parsed.entries[:limit]
    ]


def get_my_recent_posts(limit=8):
    """Topics the author already wrote about recently, so the agent avoids repeats."""
    mem = memory.load()
    posts = [d for d in mem["drafts"] if d["kind"] == "post" and d["status"] in ("approved", "published")]
    return [{"topic": d["topic"], "date": d["ts"]} for d in posts[-limit:]]


TOOL_SPECS = [
    {
        "type": "function",
        "function": {
            "name": "get_hacker_news",
            "description": "Get the current Hacker News front page stories.",
            "parameters": {
                "type": "object",
                "properties": {"limit": {"type": "integer", "description": "Number of stories, up to 30"}},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_rss",
            "description": "Get the latest entries from a named RSS feed. Feeds: " + ", ".join(FEEDS),
            "parameters": {
                "type": "object",
                "properties": {"feed": {"type": "string", "enum": list(FEEDS)}},
                "required": ["feed"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_my_recent_posts",
            "description": "List topics the author already posted about recently.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

REGISTRY = {f.__name__: f for f in (get_hacker_news, get_rss, get_my_recent_posts)}


def run_tool(name, args):
    """Run a tool by name. Errors go back to the model as text, so it can adapt."""
    try:
        result = REGISTRY[name](**args)
    except Exception as e:  # a broken feed must not crash the agent
        result = {"error": str(e)}
    return json.dumps(result)[:6000]

In [ ]:
%%writefile agent/research.py
"""Topic discovery: a hand-written tool-calling loop."""
import json

from .config import MODEL
from .llm import client, extract_json
from .tools import TOOL_SPECS, run_tool

SYSTEM = (
    "You are a research assistant for a social media manager. Use the tools to find current "
    "topics. Pick topics that fit the author's field and have not been covered recently. "
    'Finish with JSON only, in this shape: {"topics": [{"title": "", "angle": "", '
    '"why_it_fits": "", "source_url": ""}]}'
)


def discover_topics(mem, n=5, max_steps=6, verbose=True):
    messages = [
        {"role": "system", "content": SYSTEM},
        {
            "role": "user",
            "content": (
                f"Author field: {mem['field']}\n"
                f"Author background: {mem['profile'][:800]}\n"
                f"Return {n} topics. Call get_my_recent_posts first to avoid repeats."
            ),
        },
    ]
    for _ in range(max_steps):
        response = client().chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SPECS)
        message = response.choices[0].message
        if not message.tool_calls:  # the model is done
            return extract_json(message.content)["topics"]
        messages.append(message)
        for call in message.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            if verbose:
                print("tool call:", call.function.name, args)
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": run_tool(call.function.name, args)}
            )
    raise RuntimeError("The agent did not finish within max_steps")

In [ ]:
from agent import research
topics = research.discover_topics(mem)
for t in topics:
    print(t["title"], "->", t["angle"])

## 4. Post generation
Retrieve similar posts of yours, write, check with code, check with a critic, revise, save to the approval queue.

In [ ]:
%%writefile agent/generate.py
"""Post generation: retrieve examples, write, check, revise, save to the approval queue."""
import json
import re

from . import memory
from .config import PLATFORMS
from .llm import chat, chat_json
from .style import banned_hits, retrieve_examples


def system_prompt(mem, platform, kind="post"):
    p = PLATFORMS[platform]
    limit = p["max_chars"] if kind == "post" else p["comment_max_chars"]
    lessons = "\n".join("- " + x for x in mem["lessons"][-8:]) or "none yet"
    return f"""You write {platform} {kind}s as the author described below.

AUTHOR BACKGROUND:
{mem['profile'][:1000] or 'not provided'}

FIELD: {mem['field']}

STYLE PROFILE:
{json.dumps(mem['style'], indent=2)}

FAVORITE WORDS (use when natural): {', '.join(mem['favorite_words']) or 'none'}
BANNED WORDS (never use): {', '.join(mem['banned_words']) or 'none'}

LESSONS FROM PAST EDITS AND PERFORMANCE:
{lessons}

PLATFORM RULES: at most {limit} characters. {p['rules']}

HARD RULES:
- Use only facts from the source material and the author background.
- Never invent statistics, quotes or personal experiences.
- Where a personal example would help, write a placeholder in square brackets,
  like [ADD: a project where you saw this], for the author to fill in.
- Write in first person. Output only the text of the {kind}."""


def hard_checks(text, mem, max_chars):
    """Deterministic checks. These cost nothing and never disagree with themselves."""
    issues = []
    if len(text) > max_chars:
        issues.append(f"Too long: {len(text)} characters, limit {max_chars}")
    hits = banned_hits(text, mem["banned_words"])
    if hits:
        issues.append("Uses banned words: " + ", ".join(hits))
    return issues


def critique(draft, examples, source, mem):
    """LLM check against the author's real examples. Returns {'pass': bool, 'issues': [...]}"""
    prompt = (
        'Return JSON: {"pass": true or false, "issues": ["..."]}.\n'
        "Compare the DRAFT with the author's REAL EXAMPLES. Fail the draft if its tone, sentence "
        "length or structure clearly differs, if it contains buzzword filler, or if it states a "
        "specific fact that is not in the SOURCE. Placeholders in square brackets are allowed.\n\n"
        f"REAL EXAMPLES:\n{examples}\n\nSOURCE:\n{source}\n\nDRAFT:\n{draft}"
    )
    return chat_json([{"role": "user", "content": prompt}])


def write_post(mem, topic, platform="linkedin", max_revisions=2):
    query = f"{topic['title']} {topic.get('angle', '')}"
    examples = "\n\n---\n\n".join(retrieve_examples(mem, query, k=4, kind="post"))
    source = json.dumps(topic)
    messages = [
        {"role": "system", "content": system_prompt(mem, platform)},
        {
            "role": "user",
            "content": f"Real posts by the author:\n{examples}\n\nSOURCE MATERIAL:\n{source}\n\n"
                       "Write a new post about this topic.",
        },
    ]
    draft = chat(messages)
    for _ in range(max_revisions):
        issues = hard_checks(draft, mem, PLATFORMS[platform]["max_chars"])
        verdict = critique(draft, examples, source, mem)
        if not verdict.get("pass"):
            issues += verdict.get("issues", [])
        if not issues:
            break
        messages += [
            {"role": "assistant", "content": draft},
            {"role": "user", "content": "Revise the post. Fix these issues and keep everything else:\n- "
                                        + "\n- ".join(issues)},
        ]
        draft = chat(messages)
    return memory.add(mem, "drafts", {
        "kind": "post",
        "platform": platform,
        "topic": topic["title"],
        "text": draft,
        "original": draft,
        "placeholders": re.findall(r"\[[^\]]+\]", draft),
        "status": "pending",
    })

In [ ]:
from agent import generate
draft = generate.write_post(mem, topics[0])
print(draft["text"])
print("Fill in:", draft["placeholders"])

## 5. Comment drafting

In [ ]:
%%writefile agent/comment.py
"""Comment drafting: three options in the author's voice for someone else's post."""
import json

from . import memory
from .config import PLATFORMS
from .generate import hard_checks, system_prompt
from .llm import chat_json
from .style import retrieve_examples

INTENTS = [
    "add one concrete point of view",
    "ask one sharp question",
    "share a short related experience, using an [ADD: ...] placeholder instead of inventing one",
]


def draft_comments(mem, post_text, platform="linkedin"):
    examples = retrieve_examples(mem, post_text, k=4, kind="comment") or retrieve_examples(mem, post_text, k=3)
    limit = PLATFORMS[platform]["comment_max_chars"]
    prompt = (
        "Write three comments on the post below, one per intent. "
        'Return JSON: {"comments": [{"intent": "", "text": ""}]}.\n'
        "Intents:\n- " + "\n- ".join(INTENTS) + "\n\n"
        "The author's real comments and posts, for voice:\n" + "\n---\n".join(examples) + "\n\n"
        "POST TO COMMENT ON:\n" + post_text
    )
    reply = chat_json([
        {"role": "system", "content": system_prompt(mem, platform, kind="comment")},
        {"role": "user", "content": prompt},
    ])
    drafts = []
    for c in reply["comments"]:
        drafts.append(memory.add(mem, "drafts", {
            "kind": "comment",
            "platform": platform,
            "topic": "comment on: " + post_text[:60],
            "intent": c["intent"],
            "text": c["text"],
            "original": c["text"],
            "issues": hard_checks(c["text"], mem, limit),
            "status": "pending",
        }))
    return drafts

In [ ]:
from agent import comment
post = "Most teams add agents before they have an evaluation set. Here is how we build the eval first."
for c in comment.draft_comments(mem, post):
    print(c["intent"], "\n", c["text"], "\n")

## 6. Human in the loop and measurement

In [ ]:
%%writefile agent/review.py
"""Human in the loop: approve, edit or reject drafts, and learn from what the human changed."""
from . import memory
from .llm import chat_json


def approve(mem, draft_id, edited_text=None, note=""):
    draft = memory.find(mem, "drafts", draft_id)
    if edited_text and edited_text != draft["text"]:
        memory.add(mem, "feedback", {
            "draft_id": draft_id, "before": draft["text"], "after": edited_text, "note": note,
        })
        draft["text"] = edited_text
    draft["status"] = "approved"
    memory.save(mem)
    return draft


def reject(mem, draft_id, reason):
    draft = memory.find(mem, "drafts", draft_id)
    draft["status"] = "rejected"
    memory.add(mem, "feedback", {"draft_id": draft_id, "reason": reason, "before": draft["text"]})
    return draft


def learn_from_edits(mem, last=6):
    """Turn the human's edits and rejections into short writing rules."""
    edits = [f for f in mem["feedback"] if f.get("after")][-last:]
    rejections = [f for f in mem["feedback"] if f.get("reason")][-last:]
    if not edits and not rejections:
        return []
    prompt = (
        'Return JSON: {"rules": ["..."]} with at most three short, specific writing rules that '
        "explain these changes. Skip rules that only apply to one topic.\n\nEDITS (before, after):\n"
        + "\n\n".join(f"BEFORE: {e['before']}\nAFTER: {e['after']}" for e in edits)
        + "\n\nREJECTIONS:\n"
        + "\n".join(f"- {r['reason']}" for r in rejections)
    )
    rules = chat_json([{"role": "user", "content": prompt}])["rules"]
    mem["lessons"] = (mem["lessons"] + [r for r in rules if r not in mem["lessons"]])[-20:]
    memory.save(mem)
    return rules

In [ ]:
%%writefile agent/measure.py
"""Measurement: log results by hand or from a CSV, and turn them into lessons."""
import pandas as pd

from . import memory
from .llm import chat_json


def mark_published(mem, draft_id, url=""):
    draft = memory.find(mem, "drafts", draft_id)
    draft["status"] = "published"
    draft["url"] = url
    memory.save(mem)


def log_metrics(mem, draft_id, impressions, reactions, comments):
    if not memory.find(mem, "drafts", draft_id):
        raise KeyError(f"No draft with id {draft_id}")
    return memory.add(mem, "metrics", {
        "draft_id": draft_id, "impressions": impressions, "reactions": reactions, "comments": comments,
    })


def import_csv(mem, path):
    """CSV columns: draft_id, impressions, reactions, comments."""
    df = pd.read_csv(path)
    for r in df.itertuples():
        log_metrics(mem, str(r.draft_id), int(r.impressions), int(r.reactions), int(r.comments))
    return len(df)


def leaderboard(mem):
    latest = {m["draft_id"]: m for m in mem["metrics"]}  # last entry per draft wins
    rows = []
    for draft_id, m in latest.items():
        draft = memory.find(mem, "drafts", draft_id)
        rate = (m["reactions"] + m["comments"]) / max(m["impressions"], 1)
        rows.append({
            "draft_id": draft_id,
            "topic": draft["topic"],
            "impressions": m["impressions"],
            "engagement_rate": round(rate, 4),
            "text": draft["text"],
        })
    return sorted(rows, key=lambda r: -r["engagement_rate"])


def learn_from_performance(mem, top=3, min_posts=4):
    rows = leaderboard(mem)
    if len(rows) < min_posts:
        return []
    best, worst = rows[:top], rows[-top:]
    show = lambda rs: "\n\n".join(f"[{r['engagement_rate']:.2%}] {r['text']}" for r in rs)
    prompt = (
        'Return JSON: {"lessons": ["..."]} with at most three specific, testable lessons about what '
        "separates the best posts from the worst. Base them only on the texts below.\n\n"
        f"BEST:\n{show(best)}\n\nWORST:\n{show(worst)}"
    )
    lessons = chat_json([{"role": "user", "content": prompt}])["lessons"]
    mem["lessons"] = (mem["lessons"] + [x for x in lessons if x not in mem["lessons"]])[-20:]
    memory.save(mem)
    return lessons

In [ ]:
from agent import review, measure
review.approve(mem, draft["id"], edited_text=draft["text"] + "\n\n(my edit)", note="added a closing line")
measure.mark_published(mem, draft["id"])
measure.log_metrics(mem, draft["id"], impressions=1800, reactions=42, comments=7)
print(measure.leaderboard(mem)[0]["engagement_rate"])
print(review.learn_from_edits(mem))

## 7. The app
The Streamlit app wires every module together. Run it on your laptop for the final demo. Download the project with the next cell, unzip it, then run:
```
pip install -r requirements.txt
streamlit run app.py
```

In [ ]:
%%writefile app.py
"""Streamlit front end. Run with: streamlit run app.py"""
import os

import streamlit as st

from agent import comment, generate, measure, memory, research, review, style
from agent.config import PLATFORMS, SOURCES_DIR

st.set_page_config(page_title="My AI Social Media Manager", layout="wide")

with st.sidebar:
    st.header("Settings")
    key = st.text_input("OpenAI API key", type="password", value=os.getenv("OPENAI_API_KEY", ""))
    if key:
        os.environ["OPENAI_API_KEY"] = key
    platform = st.selectbox("Platform", list(PLATFORMS))

mem = memory.load()
st.title("My AI Social Media Manager")
tab_style, tab_topics, tab_post, tab_comment, tab_queue = st.tabs(
    ["1. Style", "2. Topics", "3. Post", "4. Comment", "5. Queue and results"]
)

with tab_style:
    mem["field"] = st.text_input("What do you write about?", value=mem["field"])
    files = st.file_uploader("LinkedIn export files or text files", accept_multiple_files=True,
                             type=["csv", "txt", "md"])
    fav = st.text_input("Favorite words (comma separated)", value=", ".join(mem["favorite_words"]))
    ban = st.text_input("Banned words (comma separated)", value=", ".join(mem["banned_words"]))
    if st.button("Learn my style"):
        os.makedirs(SOURCES_DIR, exist_ok=True)
        for f in files or []:
            with open(os.path.join(SOURCES_DIR, f.name), "wb") as out:
                out.write(f.getbuffer())
        style.set_words(mem, fav.split(","), ban.split(","))
        with st.spinner("Reading your writing..."):
            added = style.ingest(mem)
            style.build_profile(mem)
        st.success(f"Added {added} new items.")
    if mem["style"]:
        st.json(mem["style"])

with tab_topics:
    if st.button("Find topics"):
        with st.spinner("Checking Hacker News and RSS feeds..."):
            st.session_state["topics"] = research.discover_topics(mem, verbose=False)
    for t in st.session_state.get("topics", []):
        st.markdown(f"**{t['title']}**  \n{t['angle']}  \n_{t['why_it_fits']}_  \n{t['source_url']}")

with tab_post:
    topics = st.session_state.get("topics", [])
    if not topics:
        st.info("Find topics first.")
    else:
        choice = st.selectbox("Topic", [t["title"] for t in topics])
        if st.button("Write post"):
            topic = next(t for t in topics if t["title"] == choice)
            with st.spinner("Writing and checking..."):
                st.session_state["last_post"] = generate.write_post(mem, topic, platform)
        post = st.session_state.get("last_post")
        if post:
            st.code(post["text"], language=None)  # the copy button comes with st.code
            if post["placeholders"]:
                st.warning("Fill in before posting: " + "; ".join(post["placeholders"]))

with tab_comment:
    target = st.text_area("Paste the post you want to comment on", height=200)
    if st.button("Draft comments") and target.strip():
        with st.spinner("Drafting..."):
            st.session_state["comments"] = comment.draft_comments(mem, target, platform)
    for c in st.session_state.get("comments", []):
        st.caption(c["intent"])
        st.code(c["text"], language=None)

with tab_queue:
    mem = memory.load()
    for d in [d for d in mem["drafts"] if d["status"] == "pending"]:
        with st.expander(f"{d['kind']}: {d['topic']}"):
            edited = st.text_area("Edit", value=d["text"], key="edit" + d["id"], height=200)
            c1, c2 = st.columns(2)
            if c1.button("Approve", key="ok" + d["id"]):
                review.approve(mem, d["id"], edited)
                st.rerun()
            if c2.button("Reject", key="no" + d["id"]):
                review.reject(mem, d["id"], "rejected in UI")
                st.rerun()
    st.subheader("Log results")
    approved = [d for d in mem["drafts"] if d["status"] in ("approved", "published") and d["kind"] == "post"]
    if approved:
        pick = st.selectbox("Post", [f"{d['id']} | {d['topic']}" for d in approved])
        imp = st.number_input("Impressions", 0, step=100)
        rea = st.number_input("Reactions", 0)
        com = st.number_input("Comments", 0)
        if st.button("Save results"):
            measure.mark_published(mem, pick.split(" | ")[0])
            measure.log_metrics(mem, pick.split(" | ")[0], int(imp), int(rea), int(com))
            st.success("Saved.")
    if st.button("Learn from edits and results"):
        rules = review.learn_from_edits(mem) + measure.learn_from_performance(mem)
        st.write(rules or "Not enough data yet.")
    if mem["metrics"]:
        st.dataframe([{k: v for k, v in r.items() if k != "text"} for r in measure.leaderboard(mem)])

In [ ]:
%%writefile requirements.txt
openai>=1.50
openai-agents
feedparser
requests
numpy
pandas
streamlit

In [ ]:
import shutil, os
os.makedirs("export/smm-agent", exist_ok=True)
for item in ["agent", "sources", "app.py", "requirements.txt"]:
    dest = os.path.join("export/smm-agent", item)
    shutil.copytree(item, dest, dirs_exist_ok=True) if os.path.isdir(item) else shutil.copy(item, dest)
shutil.make_archive("smm-agent", "zip", "export")
from google.colab import files
files.download("smm-agent.zip")

## 8. Optional: the same agent with the OpenAI Agents SDK

In [ ]:
%%writefile sdk_version.py
"""The same agent built with the OpenAI Agents SDK. Run with: python sdk_version.py"""
import json

from agents import Agent, Runner, function_tool
from pydantic import BaseModel

from agent import generate, memory, tools
from agent.config import MODEL


@function_tool
def hacker_news(limit: int = 15) -> str:
    """Current Hacker News front page stories."""
    return json.dumps(tools.get_hacker_news(limit))


@function_tool
def rss(feed: str) -> str:
    """Latest entries from a configured RSS feed."""
    return json.dumps(tools.get_rss(feed))


@function_tool
def recent_posts() -> str:
    """Topics the author already posted about."""
    return json.dumps(tools.get_my_recent_posts())


class Topic(BaseModel):
    title: str
    angle: str
    why_it_fits: str
    source_url: str


class TopicList(BaseModel):
    topics: list[Topic]


researcher = Agent(
    name="Researcher",
    instructions="Find five current topics that fit the author's field and avoid recent repeats.",
    tools=[hacker_news, rss, recent_posts],
    model=MODEL,
    output_type=TopicList,
)


@function_tool
def write_post(title: str, angle: str) -> str:
    """Write a post in the author's voice and add it to the approval queue."""
    mem = memory.load()
    draft = generate.write_post(mem, {"title": title, "angle": angle})
    return draft["text"]


manager = Agent(
    name="Social media manager",
    instructions="Research topics, then write a post for the best one. Never publish anything.",
    tools=[researcher.as_tool(tool_name="research_topics", tool_description="Find fresh topics"), write_post],
    model=MODEL,
)

if __name__ == "__main__":
    mem = memory.load()
    result = Runner.run_sync(manager, f"My field is {mem['field']}. Draft one post for me.")
    print(result.final_output)

In [ ]:
!python sdk_version.py